# Name That Model - Improved Training Pipeline

This notebook uses enhanced stylometric features and ensemble methods to identify AI-generated text authorship.

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


## 1. Load Data

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Training samples: {len(train)}')
print(f'Test samples: {len(test)}')
print(f'Columns: {list(train.columns)}')
if 'model' in train.columns:
    print(f'Number of classes: {train["model"].nunique()}')
    print(train['model'].value_counts())
else:
    print('ERROR: model column not found in train.csv')

Training samples: 2699
Test samples: 3
Columns: ['id', 'prompt_id', 'text', 'model']
Number of classes: 18
model
qwen3.8-max               150
claude-sonnet-5           150
gemini-3.6-flash          150
gpt-5.6-terra             150
glm-5.3-flash             150
gemini-3.1-pro-preview    150
gemini-3.8-flash          150
claude-opus-4-6           150
gpt-5.6-luna              150
claude-opus-4-8           150
gemini-3.7-flash          150
claude-opus-5             150
kimi-k3                   150
gpt-5.5                   150
gpt-6-astra               150
glm-5.3                   150
gpt-5.6-sol               150
claude-fable-5            149
Name: count, dtype: int64


## 2. Feature Engineering - Stylometric Features

In [3]:
class StylometricFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        features = []
        for text in X:
            text = str(text)
            # Basic length features
            char_count = len(text)
            word_count = len(text.split())
            sentence_count = len(re.findall(r'[.!?]+', text)) or 1
            
            # Punctuation features
            comma_count = text.count(',')
            period_count = text.count('.')
            question_count = text.count('?')
            exclamation_count = text.count('!')
            semicolon_count = text.count(';')
            colon_count = text.count(':')
            
            # Case features
            upper_count = sum(1 for c in text if c.isupper())
            lower_count = sum(1 for c in text if c.islower())
            
            # Vocabulary richness
            words = text.lower().split()
            unique_words = len(set(words)) if words else 0
            
            features.append([
                char_count,
                word_count,
                sentence_count,
                char_count / word_count if word_count > 0 else 0,
                word_count / sentence_count if sentence_count > 0 else 0,
                comma_count / word_count if word_count > 0 else 0,
                period_count / sentence_count if sentence_count > 0 else 0,
                question_count,
                exclamation_count,
                semicolon_count,
                colon_count,
                upper_count / char_count if char_count > 0 else 0,
                unique_words / word_count if word_count > 0 else 0,
            ])
        return np.array(features)

print('Stylometric features defined')

Stylometric features defined


## 3. Train Classifier with Ensemble

In [4]:
# Encode labels
if 'model' not in train.columns:
    raise KeyError(f"Column 'model' not found. Available columns: {list(train.columns)}")
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(train['model'])

# Character n-grams (captures punctuation, casing patterns)
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 6),
    max_features=15000,
    lowercase=False,  # Keep case information
    sublinear_tf=True
)

# Word n-grams
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 4),
    max_features=12000,
    stop_words=None,  # Keep all words for style
    lowercase=True,
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

# Word n-grams with punctuation awareness (via different preprocessing)
punct_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(2, 3),
    max_features=5000,
    lowercase=True,
    sublinear_tf=True
)

# Combine all text features
text_features = FeatureUnion([
    ('char', char_vectorizer),
    ('word', word_vectorizer),
    ('punct', punct_vectorizer),
])

# Create ensemble of classifiers
clf1 = LogisticRegression(
    max_iter=3000,
    solver='saga',
    C=3.0,
    class_weight='balanced',
    random_state=42,
    multi_class='multinomial'
)

clf2 = CalibratedClassifierCV(
    LinearSVC(
        C=1.5,
        class_weight='balanced',
        max_iter=3000,
        random_state=123
    ),
    method='sigmoid',
    cv=3
)

clf3 = LogisticRegression(
    max_iter=3000,
    solver='lbfgs',
    C=5.0,
    class_weight='balanced',
    random_state=789,
    multi_class='multinomial'
)

ensemble = VotingClassifier(
    estimators=[('lr1', clf1), ('svc', clf2), ('lr2', clf3)],
    voting='soft',
    weights=[2, 2, 1]
)

# Full pipeline
pipeline = Pipeline([
    ('text_features', text_features),
    ('clf', ensemble)
])

print('Training classifier ensemble...')
pipeline.fit(train['text'].astype(str), encoded_labels)
print('Training complete')

Training classifier ensemble...


Training complete


## 4. Evaluate

In [5]:
train_preds = pipeline.predict(train['text'].astype(str))
bal_acc = balanced_accuracy_score(encoded_labels, train_preds)
acc = accuracy_score(encoded_labels, train_preds)
print(f'Accuracy: {acc:.4f}')
print(f'Balanced accuracy: {bal_acc:.4f}')

Accuracy: 1.0000
Balanced accuracy: 1.0000


## 5. Predict and Create Submission

In [6]:
# Predict on test data
test_preds = pipeline.predict(test['text'].astype(str))
test_labels = label_encoder.inverse_transform(test_preds)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test['id'].astype(str),
    'model': test_labels
})

# Ensure model names are valid
valid_models = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
submission['model'] = submission['model'].apply(lambda x: x if x in valid_models else 'gpt-5.5')

# Ensure no null values and no duplicates in id
submission = submission.drop_duplicates(subset=['id'], keep='first')
submission = submission.dropna(subset=['id', 'model'])

# Save submission as parquet with header id,model
submission.to_parquet('submission.parquet', index=False)

print('Submission created: submission.parquet')
print(submission.shape)
print(submission.to_string(index=False))

Submission created: submission.parquet
(3, 2)
          id            model
00353fb1c541 gemini-3.8-flash
005f5a9a6139 gemini-3.6-flash
0062a53d5487 gemini-3.7-flash
